# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [15]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [16]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

### Ok, I guess I need to test all combinations, so I will rearange this cells a bit:

In [17]:
def build_all_classifiers():
    lr = LinearRegression()
    lr.fit(data_set, labels)
    
    knn = KNeighborsClassifier()
    knn.fit(data_set, labels)
    
    svm = SVC()
    svm.fit(data_set, labels)
    
    dt = DecisionTreeClassifier()
    dt.fit(data_set, labels)
    
    nb = GaussianNB()
    nb.fit(data_set, labels)
    
    qda = QuadraticDiscriminantAnalysis()
    qda.fit(data_set, labels)
    
    return {"LR": lr, "KNN": knn, "SVM": svm, "DT": dt, "NB": nb, "QDA": qda}

In [18]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = DecisionTreeClassifier()
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [21]:
from itertools import combinations
all_clfs = build_all_classifiers()
best_accuracy = 0
best_combo = None
for combo_names in combinations(all_clfs.keys(), 3):
    classifiers = tuple(all_clfs[n] for n in combo_names)
    predicted = build_stacked_classifier(classifiers)
    accuracy = accuracy_score(test_labels, predicted)
    print(f"{combo_names}: {accuracy:.4f}")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_combo = combo_names
print(f"\nBest: {best_combo} -> {best_accuracy:.4f}")

('LR', 'KNN', 'SVM'): 0.1000
('LR', 'KNN', 'DT'): 0.7500
('LR', 'KNN', 'NB'): 0.7000
('LR', 'KNN', 'QDA'): 0.7500
('LR', 'SVM', 'DT'): 0.6000
('LR', 'SVM', 'NB'): 0.6000
('LR', 'SVM', 'QDA'): 0.6000
('LR', 'DT', 'NB'): 0.1000
('LR', 'DT', 'QDA'): 0.7500
('LR', 'NB', 'QDA'): 0.6000
('KNN', 'SVM', 'DT'): 0.8500
('KNN', 'SVM', 'NB'): 0.8500
('KNN', 'SVM', 'QDA'): 0.8500
('KNN', 'DT', 'NB'): 0.0000
('KNN', 'DT', 'QDA'): 0.9000
('KNN', 'NB', 'QDA'): 0.8500
('SVM', 'DT', 'NB'): 0.0500
('SVM', 'DT', 'QDA'): 0.8000
('SVM', 'NB', 'QDA'): 0.7500
('DT', 'NB', 'QDA'): 0.8500

Best: ('KNN', 'DT', 'QDA') -> 0.9000


### Now that I have my trio, I can do the rest

In [22]:
def build_classifiers():
    c1 = KNeighborsClassifier()
    c1.fit(data_set, labels)
    c2 = DecisionTreeClassifier()
    c2.fit(data_set, labels)
    c3 = QuadraticDiscriminantAnalysis()
    c3.fit(data_set, labels)
    
    return c1, c2, c3

In [25]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

0.9


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [26]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

def calculate_accuracy_vector(predicted, labels):
    result = []
    for i in range(len(predicted)):
        if predicted[i] == labels[i]:
            result.append(0)
        else:
            result.append(1)
    return result

Fill the two functions below:

In [27]:
def set_new_weights(model):
    predicted = model.predict(test_set)
    I = np.array(calculate_accuracy_vector(predicted, test_labels))
    new_weights = (1 + I) / np.sum(1 + I)
    return new_weights

Train the classifier with the code below:

In [28]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    new_clf = DecisionTreeClassifier(max_depth=1, random_state=1)
    model = new_clf.fit(X=test_set, y=test_labels, sample_weight=weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00066269 0.00132538 0.00132538 0.00066269 0.00132538 0.00066269
 0.00066269 0.00132538 0.00132538 0.00132538 0.00066269 0.00132538
 0.00066269 0.00132538 0.00066269 0.00066269 0.00066269 0.00132538
 0.00066269 0.00066269 0.00066269 0.00066269 0.00066269 0.00132538
 0.00132538 0.00132538 0.00066269 0.00066269 0.00132538 0.00066269
 0.00132538 0.00066269 0.00132538 0.00066269 0.00132538 0.00132538
 0.00132538 0.00066269 0.00132538 0.00066269 0.00066269 0.00132538
 0.00066269 0.00132538 0.00132538 0.00132538 0.00132538 0.00066269
 0.00132538 0.00132538 0.00132538 0.00066269 0.00066269 0.00066269
 0.00132538 0.00066269 0.00066269 0.00132538 0.00066269 0.00132538
 0.00132538 0.00132538 0.00132538 0.00066269 0.00066269 0.00066269
 0.00066269 0.00132538 0.00132538 0.00132538 0.00066269 0.00066269
 0.00066269 0.00132538 0.00132538 0.00132538 0.00066269 0.00066269
 0.00066269 0.00132538 0.00132538 0.00132538 0.00132538 0.00132538
 0.00132538 0.00132538 0.00132538 0.00132538 0.00066269 0.0006

Set the validation data set:

In [29]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [30]:
def get_prediction(x):
    predictions = []
    for classifier in classifiers:
        predictions.append(classifier.predict(x))
    predictions = np.array(predictions) 
    result = []
    for i in range(predictions.shape[1]):
        votes = predictions[:, i]
        values, counts = np.unique(votes, return_counts=True)
        result.append(values[np.argmax(counts)])
    return np.array(result)

Test it:

In [31]:
prediction = get_prediction(validate_x)[0]

print(prediction)

0
